In [1]:
# import some library
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("powerplant_data.csv")

df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [3]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [ ]:
# AT => temperature
# V => vacuum
# AP => pressure
# RH => humidity

# PE => produced energy

In [4]:
X = df.drop("PE", axis=1)
y = df["PE"]

In [15]:
# Convertiing dataset into tensor

# 1. split the dataset
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
    X , y , test_size=0.2 , random_state=42
)

In [20]:
# 2. Apply standardscaler
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [21]:
# Convert into tensor
import torch
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled , dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values , dtype=torch.float32).view(-1,1)

X_test_tensor = torch.tensor(X_test_scaled , dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values , dtype=torch.float32).view(-1,1)

In [25]:
# using tensor datset load in Dataloder for baching
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor , y_train_tensor )
test_dataset = TensorDataset(X_test_tensor , y_test_tensor)

In [27]:
# use dataloader add bach

train_loader = DataLoader(train_dataset , batch_size=32 , shuffle=True)
test_loader = DataLoader(test_dataset , batch_size=32)

# Build ANN Model

In [39]:
# Deep Learning
class ANN(nn.Module):
    def __init__(self):
        super(ANN , self).__init__()

        self.model = nn.Sequential(

            # 1 Hidden layer
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),

            # 2 Hidden Layer 
            nn.Linear(6, 6),
            nn.ReLU(),

            # Output Layer
            nn.Linear(6, 1),
        )

    def forward(self , x):
        return self.model(x)

In [40]:
# Build Actual Model
import torch.optim as optim
model = ANN()

# Losss & Optimzer 
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())


In [42]:
# Train Our Ann
train_losses = []
val_losses = []

epochs = 50

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # Total runn loss for epoch
    for xb , yb in train_loader:
        # xb ==> feature of one batches 
        # yb ==> labels of on batch
        optimizer.zero_grad()

        outputs = model(xb) # forward Prop>>> predict output for this
        loss = crietrion(outputs , yb) # computr loss
        loss.backward() # back....propogation compute gradient
        optimizer.step() # param update

        running_loss += loss.item() # loss is tensor to ==> py float

    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    # Viladition in looop epochs 
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad(): #no gradient compute
        for xb , yb in test_loader:
            outputs = model(xb)
            loss = crietrion(outputs , yb)
            running_val_loss += loss

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"epoch{epoch+1} / {epochs} ==> training loss = {epoch_train_loss} & loss {running_val_loss} ")

    

epoch1 / 50 ==> training loss = 186152.63209635418 & loss 9970540.0 
epoch2 / 50 ==> training loss = 138039.06279296876 & loss 6552542.0 
epoch3 / 50 ==> training loss = 85818.2021484375 & loss 4055044.5 
epoch4 / 50 ==> training loss = 54623.55080566406 & loss 2633330.75 
epoch5 / 50 ==> training loss = 33721.10065104167 & loss 1487756.5 
epoch6 / 50 ==> training loss = 17571.20216878255 & loss 714591.125 
epoch7 / 50 ==> training loss = 8660.208826700846 & loss 396772.6875 
epoch8 / 50 ==> training loss = 5486.679885864258 & loss 282339.6875 
epoch9 / 50 ==> training loss = 3981.71960144043 & loss 209218.765625 
epoch10 / 50 ==> training loss = 2955.7351755777995 & loss 158817.0625 
epoch11 / 50 ==> training loss = 2267.633920796712 & loss 125352.2109375 
epoch12 / 50 ==> training loss = 1805.9344429016114 & loss 101938.109375 
epoch13 / 50 ==> training loss = 1478.4522232055665 & loss 84429.265625 
epoch14 / 50 ==> training loss = 1240.6314969380696 & loss 70758.3046875 
epoch15 / 5

In [43]:
# Plot the model 

loss_df = pd.DataFrame({
    "Training Loss" : train_losses,
    "Validation Loss" : val_losses
})

plt.plot(loss_df["Training Loss"] , label="Training loss")
plt.plot(loss_df["Validation Loss"] , label="Validation Loss")

plt.xlabel("Epoch")
plt.xlabel("Losses")
plt.legend()

NameError: name 'plt' is not defined